# 🚀 00 — Environment Setup
## Ekegusii-LLM-Translation · Kineses Cloud / Base Jupyter

> **Run this notebook ONCE before opening any other notebook.**

### ⚠️ IMPORTANT — Why `!pip install` FAILS on Kineses / Base Jupyter

| Command | What it does | Result |
|---------|-------------|--------|
| `!pip install pandas` | Installs into **system Python** (not your Jupyter kernel) | ❌ `ModuleNotFoundError` persists |
| `!python -m pip install pandas` | `python` may point to wrong binary | ❌ Unreliable |
| `sys.executable` method ✅ | Installs into the **exact Python that IS this notebook** | ✅ Always works |

---
| Node Spec | Value |
|-----------|-------|
| CPU Cores | 22 |
| RAM | 117.9 GB |
| Disk | 967.64 GB |
| Target GPU | NVIDIA A100 80GB |

In [ ]:
# ==============================================================
# MASTER BOOTSTRAP CELL
# Step 1: Install ALL packages using sys.executable (FIXES ModuleNotFoundError)
# Step 2: Download repository ZIP (no git binary needed)
# Step 3: Set working directory & sys.path
# Step 4: Verify corpus & hardware
#
# CRITICAL: This uses sys.executable — NOT !pip — to guarantee
# packages install into the EXACT Python kernel running this cell.
# ==============================================================

import sys
import subprocess

print(f"Python kernel: {sys.executable}")
print(f"Python version: {sys.version}")
print("\n" + "="*65)
print("STEP 1/4 — Installing packages into THIS kernel")
print("="*65)

# ---------------------------------------------------------------
# ALL packages installed via sys.executable — never !pip install
# ---------------------------------------------------------------
ALL_PACKAGES = [
    # Core data science
    "pandas", "numpy", "matplotlib", "seaborn", "plotly",
    "scipy", "scikit-learn",
    # HuggingFace ecosystem
    "transformers", "datasets", "evaluate", "tokenizers",
    "accelerate", "huggingface_hub",
    # QLoRA stack
    "peft", "trl", "bitsandbytes",
    # Evaluation
    "sacrebleu", "unbabel-comet", "nltk",
    # Config & CLI
    "rich", "typer", "hydra-core", "omegaconf",
    # Utilities
    "tqdm", "ipywidgets",
]

for pkg in ALL_PACKAGES:
    print(f"  Installing {pkg}...", end=" ")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅")
    else:
        print(f"❌ FAILED: {result.stderr.strip()[:80]}")

print("\n" + "="*65)
print("STEP 2/4 — Downloading repository ZIP (no git needed)")
print("="*65)

import urllib.request
import zipfile
import os

REPO_URL = "https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip"
ZIP_NAME = "repo.zip"
PROJ_DIR = "Ekegusii-LLM-Translation-main"

if not os.path.isdir(PROJ_DIR):
    print("📥 Downloading (~15 MB)...")
    urllib.request.urlretrieve(REPO_URL, ZIP_NAME)
    print("📦 Extracting files...")
    with zipfile.ZipFile(ZIP_NAME, "r") as z:
        z.extractall(".")
    os.remove(ZIP_NAME)
    print(f"✅ Repository extracted to: ./{PROJ_DIR}/")
else:
    print(f"✅ Repository already present — skipping download.")

print("\n" + "="*65)
print("STEP 3/4 — Setting working directory & import paths")
print("="*65)

# Navigate into project root
if os.path.basename(os.getcwd()) != PROJ_DIR:
    os.chdir(PROJ_DIR)

# Add root to sys.path so 'from src.xxx import yyy' works everywhere
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📁 Working Directory : {os.getcwd()}")
print(f"🐍 Python Kernel     : {sys.executable}")

# Verify folder structure
REQUIRED_DIRS = [
    "src", "data", "notebooks", "configs",
    "tests", "scripts",
    "data/master_corpus",
    "src/master_corpus",
    "src/task_generation",
    "src/evaluation",
    "src/models",
]
all_ok = True
for d in REQUIRED_DIRS:
    exists = os.path.isdir(d)
    if not exists:
        all_ok = False
    print(f"  {'✅' if exists else '❌'}  {d}/")

print("\n" + "="*65)
print("STEP 4/4 — Verifying imports, hardware, and corpus")
print("="*65)

import torch
import pandas as pd
import numpy as np
import transformers
import peft
import sacrebleu

print(f"  PyTorch      : {torch.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  NumPy        : {np.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  PEFT         : {peft.__version__}")
print(f"  SacreBLEU    : {sacrebleu.__version__}")
print(f"  GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"  GPU VRAM     : {props.total_memory / 1e9:.1f} GB")
else:
    print("  ⚠️  No GPU detected — training will use CPU.")

from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.integrity import DataLeakageChecker

manager = MasterCorpusManager()
corpus  = manager.load_sentence_corpus()
lexical = manager.load_lexical_corpus()
train   = manager.load_train_split()
val     = manager.load_val_split()
test    = manager.load_test_split()

print(f"\n  Master Sentence Corpus : {len(corpus):,} multilingual concepts")
print(f"  Master Lexical Corpus  : {len(lexical):,} dictionary entries")
print(f"  Train Split            : {len(train):,} (80%)")
print(f"  Val   Split            : {len(val):,} (10%)")
print(f"  Test  Split            : {len(test):,} (10%)")

checker = DataLeakageChecker(manager)
checker.verify_all()

print("\n" + "="*65)
print("✅  0% DATA LEAKAGE CONFIRMED")
print("✅  ALL IMPORTS WORKING")
print("✅  SETUP COMPLETE — OPEN ANY NOTEBOOK IN notebooks/")
print("="*65)
print("\n📋 NEXT STEPS:")
print("  → notebooks/05_instruction_generation.ipynb")
print("  → notebooks/07_train_aya.ipynb")
print("  → notebooks/08_train_llama.ipynb")
print("  → notebooks/09_translation_evaluation.ipynb")